In [ ]:
%pip install requests

In [1]:
import sqlite3

# Connect to the db
conn = sqlite3.connect('travel_planner.db')

# Create the cursor
cursor = conn.cursor()

# Now we are good to go, let's add a VibeName (Notice the coma? we must add it so the thing become a tuple)
new_vibe = ("Budget backpacking",)

# Now we execute the change using the cursor
cursor.execute("INSERT INTO Travel_Vibes (VibeName) VALUES (?)", new_vibe)

# That's done, now let's add multiple values at once:

cities_to_add = [
    ("Tokyo", "Japan", "The capital of Japan, a beautiful place to visit anytime."),
    ("Bucharest", "Romania", "The capital of Romania, it's alright."),
    ("London", "England", "The capital of England, it's a nice place for pubcrawling.")
]

# Now bcs we have multiple arrays, we need to use a loop to add everything
cursor.executemany("""
    INSERT INTO Destinations (CityName, Country, Description)
    VALUES (?, ?, ?)
""", cities_to_add)

# Everything is done, let's commit our changes

conn.commit()
print("Data succesfully saved!")

# Now we close it
conn.close()

Data succesfully saved!


In [2]:
# Let's print some data

# Connect to the db
conn = sqlite3.connect('travel_planner.db')

# Create the cursor
cursor = conn.cursor()

# Ask the cursor to get all columns (*) from Destinations
cursor.execute("SELECT * FROM Destinations")

# Actually grab the results
all_destinations = cursor.fetchall()

# The results come back as a list of tuples: [(1, 'Tokyo', 'Japan', '...'), (2, 'Kyoto', ...)]
print("Here are your cities:")
for city in all_destinations:
    print(f"ID: {city[0]} | City: {city[1]} | Country: {city[2]}")

# Now we close it
conn.close()

Here are your cities:
ID: 1 | City: Tokyo | Country: Japan
ID: 2 | City: Bucharest | Country: Romania
ID: 3 | City: London | Country: England


In [11]:
# Let's update the data shall we?

# Connect to the db
conn = sqlite3.connect('travel_planner.db')

# Create the cursor
cursor = conn.cursor()

# Let's change the description for Bucharest, we know it has an id of 2
update_data = ("It has one of the biggest buildings in the world.", 2)
update_data = ("A very beautiful city, one of the most beautiful cities in the world!", 3)

cursor.execute("""
    UPDATE Destinations 
    SET Description = ? 
    WHERE DestinationID = ?
""", update_data)
conn.commit()

# --- DELETING DATA ---
# Let's delete Tokyo
cursor.execute("DELETE FROM Destinations WHERE DestinationID = ?", (1,))
conn.commit()

conn.close()

In [12]:
import sqlite3

# Connect to the db
conn = sqlite3.connect('travel_planner.db')

# Create the cursor
cursor = conn.cursor()

# ---> THIS IS THE MISSING LINE! Tell the cursor what to grab <---
cursor.execute("SELECT * FROM Destinations")

# Now it actually has data to fetch
all_destinations = cursor.fetchall()

# Print the basic info
print("--- City Info ---")
for city in all_destinations:
    print(f"ID: {city[0]} | City: {city[1]} | Country: {city[2]}")

print("\n--- Descriptions ---")
# Small fix here too: description is at index 3! 
# Indexes: 0=ID, 1=CityName, 2=Country, 3=Description
for row in all_destinations:
    print(f"City: {row[1]} | Description: {row[3]}")

conn.close()

--- City Info ---
ID: 2 | City: Bucharest | Country: Romania
ID: 3 | City: London | Country: England

--- Descriptions ---
City: Bucharest | Description: It has one of the biggest buildings in the world.
City: London | Description: A very beautiful city, one of the most beautiful cities in the world!


In [3]:
# Let's use panda to visualise all the data

import sqlite3
import pandas as pd
from IPython.display import display # This lets us render multiple beautiful tables

# Open the connection
conn = sqlite3.connect('travel_planner.db')

# Ask SQLite's master directory for the names of all your tables
master_query = "SELECT name FROM sqlite_master WHERE type='table';"

# Put those names into a pandas DataFrame
tables_df = pd.read_sql_query(master_query, conn)

print("Here is the list of tables in your database:")
display(tables_df)

print("\n--- Displaying contents of each table ---")

# Loop through that list of names
for table_name in tables_df['name']:
    print(f"\nData for table: {table_name}")
    
    # Write a new query for each specific table using an f-string
    query = f"SELECT * FROM {table_name}"
    
    # Fetch the data and display it
    df = pd.read_sql_query(query, conn)
    display(df)

# Pagma
# Ask SQLite for the foreign key rules of your link table
print("Here are the rules of our tables")
pragma_query = "PRAGMA foreign_key_list('Destinations_Activities');"

# Read it into pandas
df_keys = pd.read_sql_query(pragma_query, conn)

conn.close()

df_keys

Here is the list of tables in your database:


,name
0,Destinations
1,Activities
2,Travel_Vibes
3,Destinations_Activities
4,Destination_Vibes



--- Displaying contents of each table ---

Data for table: Destinations


,DestinationID,CityName,Country,Description
0,1,Tokyo,Japan,"The capital of Japan, a beautiful place to vis..."
1,2,Bucharest,Romania,"The capital of Romania, it's alright."
2,3,London,England,"The capital of England, it's a nice place for ..."
3,4,Paris,France,"The capital of France, iconic for food, art, a..."
4,5,Rome,Italy,"The capital of Italy, packed with ancient hist..."
5,6,New York,USA,"The Big Apple, famous for its non-stop energy ..."
6,7,Barcelona,Spain,A vibrant coastal city known for stunning arch...
7,8,Bangkok,Thailand,A bustling tropical hub famous for street food...
8,9,Sydney,Australia,A gorgeous harbor city with amazing beaches an...
9,10,Cairo,Egypt,A historic desert metropolis home to the ancie...



Data for table: Activities


...

,DestinationID,ActivityID,Spotlight_Description
0,1,9,Eat your way through the Tsukiji Outer Market.
1,4,10,Spend days getting lost in the Louvre and Musé...
2,8,1,Backpack through Khao San Road for the ultimat...



Data for table: Destination_Vibes


,DestinationID,VibeID
0,1,7
1,1,9
2,2,1
3,2,7
4,3,7
5,3,10
6,4,9
7,4,10
8,5,9
9,5,10


Here are the rules of our tables


,id,seq,table,from,to,on_update,on_delete,match
0,0,0,Activities,ActivityID,ActivityID,NO ACTION,NO ACTION,NONE
1,1,0,Destinations,DestinationID,DestinationID,NO ACTION,NO ACTION,NONE
